# Building the SOPP Dataset

In [1]:
# packages
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
# from shapely.geometry import Point

In [2]:
sopp_df = pd.read_csv("sopp_df_3_8.csv")

/tmp/ipykernel_34730/1893267037.py:1: DtypeWarning: Columns (8,10,17,18,19,24,25,26,27,31,32,33,34,35,37,38,39,40,42,43) have mixed types. Specify dtype option on import or set low_memory=False.
  sopp_df = pd.read_csv("sopp_df_3_8.csv")


## Implementing Precinct Names and Time Segments (Copied from Rachel's Code)

In [3]:
sopp_df.head()

,Unnamed: 0.1,Unnamed: 0,raw_row_number,date,time,location,lat,lng,precinct,reporting_area,...,MILEPT,LGT_COND,VE_TOTAL,SP_JUR,HOSP_MN,VE_FORMS,NHS,ARR_MIN,DAY,cluster_label
0,1,1,237161,2010-10-10,10:00:00,"1122 LEBANON PIKE, NASHVILLE, TN, 37210",36.155521,-86.735902,5.0,9035.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,5,5,232748,2010-10-10,01:00:00,"N GALLATIN PIKE & EDENWOLD RD, MADISON, TN, 37115",36.286844,-86.705395,7.0,1757.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
2,6,6,305651,2010-10-10,22:02:00,"1 24 W, , TN,",36.157439,-86.770472,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,8,8,232900,2010-10-10,10:05:00,"HARDING PL & SHADESCREST DR, NASHVILLE, , 37211",36.077050,-86.736633,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,10,10,305803,2010-10-10,10:07:00,"I 24 W & I 65 N, NASHVILLE, TN, 37207",36.228301,-86.779231,2.0,19060.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2


In [4]:
# Drop the original precinct column
sopp_df = sopp_df.drop(columns=['precinct'])

In [5]:
from shapely.geometry import Point

# load precinct map
precincts_map = gpd.read_file('/precinct_zip.zip')

geometry = [Point(xy) for xy in zip(sopp_df['lng'], sopp_df['lat'])]
crashes_gdf = gpd.GeoDataFrame(sopp_df, geometry=geometry)

# epsg=4326 reads lat-lon
crashes_gdf.set_crs(epsg=4326, inplace=True)
precincts_map = precincts_map.to_crs(epsg=4326)

# perform spatial join on shape data and crash data
precinct_df = gpd.sjoin(crashes_gdf, precincts_map, how='left', predicate='within')

In [6]:
precinct_df.head()

,Unnamed: 0.1,Unnamed: 0,raw_row_number,date,time,location,lat,lng,reporting_area,zone,...,SP_JUR,HOSP_MN,VE_FORMS,NHS,ARR_MIN,DAY,cluster_label,index_right,PrecinctNa,GLOBALID
0,1,1,237161,2010-10-10,10:00:00,"1122 LEBANON PIKE, NASHVILLE, TN, 37210",36.155521,-86.735902,9035.0,513.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1,7.0,HERMITAGE,3c8dbffb-01f5-4852-94d8-efcfb93b8f11
1,5,5,232748,2010-10-10,01:00:00,"N GALLATIN PIKE & EDENWOLD RD, MADISON, TN, 37115",36.286844,-86.705395,1757.0,727.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2,1.0,MADISON,8496aef9-34ad-4c35-b65a-1556332686ed
2,6,6,305651,2010-10-10,22:02:00,"1 24 W, , TN,",36.157439,-86.770472,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,8.0,CENTRAL,bd3522ab-ee43-4c13-81a4-20b6081156be
3,8,8,232900,2010-10-10,10:05:00,"HARDING PL & SHADESCREST DR, NASHVILLE, , 37211",36.077050,-86.736633,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,5.0,MIDTOWN HILLS,517c3325-2df7-4dfa-b41b-b57269031e66
4,10,10,305803,2010-10-10,10:07:00,"I 24 W & I 65 N, NASHVILLE, TN, 37207",36.228301,-86.779231,19060.0,221.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2,1.0,MADISON,8496aef9-34ad-4c35-b65a-1556332686ed


In [7]:
precinct_df = precinct_df.rename(columns = {"PrecinctNa": "precinct"})

In [8]:
precinct_df[precinct_df['precinct'].isna()].head()

,Unnamed: 0.1,Unnamed: 0,raw_row_number,date,time,location,lat,lng,reporting_area,zone,...,SP_JUR,HOSP_MN,VE_FORMS,NHS,ARR_MIN,DAY,cluster_label,index_right,precinct,GLOBALID
23383,48053,51490,1899577,2014-10-16,10:30:00,"419 BATTLE RD, NOLENSVILLE, TN, 37135",35.974736,-86.637706,8771.0,333.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
23837,49078,52626,1899217,2014-10-16,09:55:00,"419 BATTLE RD, NOLENSVILLE, TN, 37135",35.974736,-86.637706,8771.0,333.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
39654,80145,85921,2276491,2015-10-21,10:07:00,"419 BATTLE RD, NOLENSVILLE, TN, 37135",35.974736,-86.637706,8771.0,333.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
39655,80147,85923,2276439,2015-10-21,10:10:00,"419 BATTLE RD, NOLENSVILLE, TN, 37135",35.974736,-86.637706,8771.0,333.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
39674,80168,85945,2276452,2015-10-21,10:20:00,"419 BATTLE RD, NOLENSVILLE, TN, 37135",35.974736,-86.637706,8771.0,333.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN


In [9]:
precinct_df = precinct_df.dropna(subset=['precinct'])
precinct_df.head()

,Unnamed: 0.1,Unnamed: 0,raw_row_number,date,time,location,lat,lng,reporting_area,zone,...,SP_JUR,HOSP_MN,VE_FORMS,NHS,ARR_MIN,DAY,cluster_label,index_right,precinct,GLOBALID
0,1,1,237161,2010-10-10,10:00:00,"1122 LEBANON PIKE, NASHVILLE, TN, 37210",36.155521,-86.735902,9035.0,513.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1,7.0,HERMITAGE,3c8dbffb-01f5-4852-94d8-efcfb93b8f11
1,5,5,232748,2010-10-10,01:00:00,"N GALLATIN PIKE & EDENWOLD RD, MADISON, TN, 37115",36.286844,-86.705395,1757.0,727.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2,1.0,MADISON,8496aef9-34ad-4c35-b65a-1556332686ed
2,6,6,305651,2010-10-10,22:02:00,"1 24 W, , TN,",36.157439,-86.770472,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,8.0,CENTRAL,bd3522ab-ee43-4c13-81a4-20b6081156be
3,8,8,232900,2010-10-10,10:05:00,"HARDING PL & SHADESCREST DR, NASHVILLE, , 37211",36.077050,-86.736633,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,5.0,MIDTOWN HILLS,517c3325-2df7-4dfa-b41b-b57269031e66
4,10,10,305803,2010-10-10,10:07:00,"I 24 W & I 65 N, NASHVILLE, TN, 37207",36.228301,-86.779231,19060.0,221.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2,1.0,MADISON,8496aef9-34ad-4c35-b65a-1556332686ed


In [10]:
# Remake the DAY_WEEK, YEAR, HOUR, MONTH Columns
precinct_df['datetime'] = pd.to_datetime(
    precinct_df['date'].astype(str) + ' ' + precinct_df['time'].astype(str),
    errors='coerce'
)

precinct_df['YEAR'] = precinct_df['datetime'].dt.year
precinct_df['MONTH'] = precinct_df['datetime'].dt.month
precinct_df['HOUR'] = precinct_df['datetime'].dt.hour
precinct_df['DAY_WEEK'] = precinct_df['datetime'].dt.dayofweek

precinct_df = precinct_df.drop(columns=[
    'Unnamed: 0',
    'Unnamed: 0.1',
    'index_right',
    'geometry',
    'time'
], errors='ignore')

In [11]:
precinct_df[["date", "MONTH"]]

,date,MONTH
0,2010-10-10,10.0
1,2010-10-10,10.0
2,2010-10-10,10.0
3,2010-10-10,10.0
4,2010-10-10,10.0
...,...,...
1392452,2016-09-09,9.0
1392453,2016-09-09,9.0
1392454,2016-09-09,9.0
1392455,2016-09-09,9.0


## Time Segments and Inflated DF

In [12]:
import itertools

# Assign Time Segments
precinct_df = precinct_df[precinct_df['HOUR'] < 24].copy()

# Map hours into segments
segment_map = {
    0: 'Late Night', 1: 'Late Night', 2: 'Late Night', 3: 'Late Night',
    4: 'Early Morning', 5: 'Early Morning', 6: 'Early Morning', 7: 'Early Morning',
    8: 'Morning', 9: 'Morning', 10: 'Morning', 11: 'Morning',
    12: 'Afternoon', 13: 'Afternoon', 14: 'Afternoon', 15: 'Afternoon',
    16: 'Evening', 17: 'Evening', 18: 'Evening', 19: 'Evening',
    20: 'Night', 21: 'Night', 22: 'Night', 23: 'Night'
}
precinct_df['time_segment'] = precinct_df['HOUR'].map(segment_map)

In [13]:
# Drop Rows with no date
precinct_df = precinct_df.dropna(subset=['date'])

In [14]:
precinct_df['date']

,date
0,2010-10-10
1,2010-10-10
2,2010-10-10
3,2010-10-10
4,2010-10-10
...,...
1392452,2016-09-09
1392453,2016-09-09
1392454,2016-09-09
1392455,2016-09-09


In [15]:
# Mark each crash
precinct_df['STOP'] = 1

# Build full grid
all_dates = pd.date_range(start=precinct_df['date'].min(), end=precinct_df['date'].max(), freq='D')
segments = ['Late Night', 'Early Morning', 'Morning', 'Afternoon', 'Evening', 'Night']
precincts = sorted(precinct_df['precinct'].dropna().unique())

master_grid = pd.DataFrame(
    itertools.product(all_dates, segments, precincts),
    columns=['date', 'time_segment', 'precinct']
)

master_grid['DAY_WEEK'] = master_grid['date'].dt.day_name()

precinct_df['date'] = pd.to_datetime(precinct_df['date'], errors='coerce')

# Count crashes per block
crash_counts = (
    precinct_df.groupby(['date', 'time_segment', 'precinct'])['STOP']
    .size()
    .reset_index()
)

# Merge onto full grid
inflated_df = master_grid.merge(
    crash_counts,
    on=['date', 'time_segment', 'precinct'],
    how='left'
)

# Fill missing blocks with zero crashes
inflated_df['STOP'] = inflated_df['STOP'].fillna(0).astype(int)

In [17]:
inflated_df['STOP'] = (inflated_df['STOP'] > 0).astype(int)

In [18]:
inflated_df.head()

,date,time_segment,precinct,DAY_WEEK,STOP
0,2010-01-01,Late Night,CENTRAL,Friday,1
1,2010-01-01,Late Night,EAST,Friday,1
2,2010-01-01,Late Night,HERMITAGE,Friday,1
3,2010-01-01,Late Night,MADISON,Friday,1
4,2010-01-01,Late Night,MIDTOWN HILLS,Friday,1


In [19]:
inflated_df.to_csv("sopp_inflated.csv", index=False)